![](https://live.staticflickr.com/65535/54531085276_a4c5d63f4f_b.jpg)

*Zdjęcia zostały wygenerowane za pomocą modelu Flux-dev, a następnie poddane edycji przez autora zadania.*

# Wstęp
Tematem niniejszego zadania jest zagadnienie poprawy jakości zdjęć (ang. image enhancement), będące pojęciem zbiorczym obejmującym takie działania jak:

- zwiększanie rozdzielczości (ang. image super-resolution),
- odszumianie (ang. image denoising),
- wyostrzanie (ang. image deblurring),
- rozjaśnianie (ang. low-light image enhancement),
- **kolorowanie** (ang. image colorization),
- oraz wiele innych.

Celem tych technik jest uzyskanie obrazu o wysokiej jakości na podstawie zdjęcia o niskiej jakości. Proces ten często wymaga uzupełnienia brakujących informacji w obrazie na podstawie jego kontekstu, dlatego powszechnie stosuje się metody uczenia głębokiego.

Dzięki takim technikom możemy w pewnym sensie „spojrzeć” w przeszłość z nowej perspektywy. Przykładem może być poniższe zdjęcie Warszawy z początku dwudziestolecia międzywojennego – po lewej w oryginale, a po prawej w wersji pokolorowanej przez Mariusza Zająca.

![](https://live.staticflickr.com/65535/54531426695_24b5710613_b.jpg)

W tym zadaniu skupimy się na kolorowaniu czarno-białych fotografii przedstawiających ludzkie twarze.

### Kolorowanie zdjęć
Zdjęcie kolorowe opisane jest trzema kanałami RGB (czerwony – red, zielony – green, niebieski – blue). Można je w prosty sposób przekształcić w obraz monochromatyczny, opisany jednym kanałem. Jednak przy konwersji do skali szarości nie stosuje się prostej średniej arytmetycznej z wartości RGB, ponieważ nie odpowiada to sposobowi, w jaki ludzkie oko postrzega jasność poszczególnych barw (np. światło niebieskie postrzegamy jako ciemniejsze niż zielone).
Z tego względu piksele w obrazie w skali szarości obliczane są według wzoru:

$$ x^{(gray)} = 0.299 \cdot x^{(red)} + 0.587 \cdot x^{(green)} + 0.114 \cdot x^{(blue)}. $$

Warto zauważyć, że konwersja z przestrzeni RGB do skali szarości jest jednoznaczna i łatwa do wykonania. Odwrotna operacja – odzyskiwanie informacji o kolorach – nie jest jednoznaczna, ponieważ wiele różnych kombinacji wartości RGB może prowadzić do tej samej wartości w skali szarości. Oznacza to, że kolorowanie obrazów jest zadaniem nieokreślonym (ang. ill-posed), co implikuje, że dla jednego czarno-białego zdjęcia można wygenerować wiele poprawnych wersji kolorowych.

Na przykład, czarno-białe zdjęcie samochodu trudno jednoznacznie pokolorować – pojazd mógł mieć niemal dowolny kolor. Z drugiej strony, modele uczone na odpowiednich danych są w stanie wykorzystać statystyczne regularności – takie jak to, że trawa zwykle jest zielona, niebo niebieskie, a tygrys pomarańczowo-czarny – aby generować najbardziej prawdopodobne wersje kolorystyczne.

Formalnie, zadanie kolorowania definiujemy jako naukę modelu $\mathcal{M}_\theta$ o parametrach $\theta$, który na wejściu otrzymuje obraz w skali szarości $x^{(gray)}$, a na wyjściu generuje jego kolorową wersję $\hat{x}^{(rgb)}$:

$$ \hat{x}^{(rgb)} = \mathcal{M}_\theta(x^{(gray)}). $$

Chcielibyśmy, aby predykcje modelu $\hat{x}^{(rgb)}$ jak najbardziej przypominały oryginalne obrazy kolorowe $x^{(rgb)}$. W tym celu dążymy do znalezienia takich wartości parametrów $\theta$, które minimalizują pewną miarę różnicy (np. błąd średniokwadratowy) pomiędzy predykcjami a rzeczywistymi obrazami:

$$ \theta^* = \arg \min_{\theta} \sum_{i = 1}^N d(x^{(rgb)}_i, \mathcal{M}_\theta(x^{(gray)}_i)), $$

gdzie $N$ oznacza liczbę przykładów w zbiorze treningowym.

### Generatywne sieci przeciwstawne 
Generatywne sieci przeciwstawne (ang. generative adversarial networks, GAN) były jednym z pierwszych podejść adresujących problem generowania danych z użyciem technik uczenia głębokiego. Ponieważ proces generowania nie wymaga żadnego wejścia do sieci, to nie da się w prosty sposób zaproponować funkcji straty, ponieważ nie wiemy jakiego wyjścia z sieci się spodziewać. Na przykład sieć może wygenerować wiarygodny obraz twarzy, ale gdy porównamy go z innym losowym zdjęciem twarzy, to kara będzie bardzo duża, mimo poprawności wyjścia sieci.

Z tego powodu architektura GAN składa się z dwóch sieci: generatora oraz dyskryminatora.

- Rolą generatora jest generowanie nowych, losowych zdjęć.
- Rolą dyskryminatora jest przewidywanie, czy wejściowy obraz pochodzi ze zbioru danych, czy został wygenerowany przez generator.

Generator stara się oszukać dyskryminator (maksymalizuje błąd klasyfikacji dyskryminatora), natomiast dyskryminator stara się zgadnąć czy obraz jest sztuczny czy nie (minimalizuje swój błąd klasyfikacji). Obie sieci uczy się naprzemiennie, aż do uzyskania zbieżności.

W tym zadaniu nie będziemy się zajmować trenowaniem GANa, natomiast będziemy mieć do dyspozycji wytrenowany wcześniej model generatora, który na podstawie szumu potrafi wygenerować losowy obraz twarzy o rozdzielczości $256 \times 256$. Architektura tej sieci nazywa się StyleGAN i zaprezentowana jest na schemacie poniżej. Dzieli się ona na dwa moduły: sieci mapującej $f$ oraz generującej $g$. 

Sieć $f$ to architektura MLP transformująca losowy wektor $z$, posiadający $512$ elementów z wartościami pochodzącymi z rozkładu normalnego $\mathcal{N}(0, \mathbb{I})$, w $512$ wymiarowy wektor $w$, który jest używany do warunkowania sieci $g$. Wektor $w$ możemy interpretować jako reprezentację ukrytą generowanego zdjęcia.

Sieć $g$ to sieć konwolucyjna o stałym wejściu $4 \times 4$, która progresywnie zwiększa jego rozdzielczość aż do $256 \times 256$. W każdym bloku, sieć $g$ jest warunkowana wektorem $w$, zapewniającym różnorodność generowanych zdjęć. Dodatkowo do sieci trafia także trochę szumu, który jeszcze bardziej zwiększa różnorodność.

![](https://live.staticflickr.com/65535/54531085226_a4fe9a0c2e_z.jpg)


# Zadanie
W tym zadaniu, mając do dyspozycji wytrenowany `generator`, potrafiący generować zdjęcia twarzy o rozdzielczości $256 \times 256$, należy zaproponować metodę do kolorowania czarno-białych zdjęć twarzy o tej samej rozdzielczości. Celem jest tutaj wyekstrahowanie wiedzy, która drzemie w wagach generatora i użycie jej w zupełnie innym zadaniu, nieznanym dotąd generatorowi. 

### Dane
W tym zadaniu nie ma dostępnych danych treningowych, a jedynie dane walidacyjne do wstępnej oceny zaproponowanego przez Ciebie podejścia. Dane walidacyjne posiadają 500 sparowanych zdjęć czarno-białych z ich kolorowymi odpowiednikami.

### Kryterium oceny
W celu oceny jakości Twojego rozwiązania, będzie użyta metryka składająca się z dwóch podmetryk: *PSNR* oraz *LPIPS*. 

PSNR jest odwrotnie proporcjonalna do błędu średniokwadratowego pomiędzy generowaną próbką $\hat{x}^{(rgb)}$, a kolorowym obrazem prawdziwym $x^{(rgb)}$. Wyraża się wzorem
$$ PSNR = 10 \cdot \log_{10}\left(\frac{\text{range}^2}{MSE(\hat{x}^{(rgb)}, x^{(rgb)})}\right), $$
gdzie $\text{range}$ to przedział możliwych wartości jakie mogą przyjmować obrazy $\hat{x}^{(rgb)}$ oraz $x^{(rgb)}$. PSNR chcemy maksymalizować. 
Punktowany zakres wartości tej metryki to $(22.0, 26.0)$ - punkty naliczane są proporcjonalnie.

LPIPS to średnia odległości $L1$ pomiędzy mapami cech po wybranych warstwach sieci `AlexNet`, uzyskanymi na podstawie wejść $\hat{x}^{(rgb)}$ oraz $x^{(rgb)}$. LPIPS chcemy minimalizować. 
Punktowany zakres wartości tej metryki to $(0.15, 0.11)$ - punkty naliczane są proporcjonalnie. 

Ostateczna wartość punktowa to średnia ważona punktów uzyskanych z metryki PSNR oraz LPIPS o wagach $0.25$ (PSNR) oraz $0.75$ (LPIPS).

### Ograniczenia

- Twoje rozwiazanie będzie testowane na Platformie Konkursowej bez dostępu do internetu oraz w środowisku z GPU.
- Ewaluacja Twojego finalnego rozwiązania na Platformie Konkursowej nie może trwać dłużej niż 5 minut z GPU.
- Lista dopuszczalnych bibliotek: torch, numpy, torchvision, pillow.

### Pliki zgłoszeniowe
Należy przesłać tylko ten notebook uzupełniony o Twoje rozwiązanie (patrz klasa `YourModel`).

### Ewaluacja
Podczas sprawdzania flaga `FINAL_EVALUATION_MODE` zostanie ustawiona na `True`.

Za to zadanie możesz zdobyć pomiędzy 0 a 100 punktów. Liczba punktów, którą zdobędziesz, będzie wyliczona na (tajnym) zbiorze testowym na Platformie Konkursowej na podstawie wyżej wspomnianego wzoru, zaokrąglona do liczby całkowitej. Jeśli Twoje rozwiązanie nie będzie spełniało powyższych kryteriów lub nie będzie wykonywać się prawidłowo, otrzymasz za zadanie 0 punktów.

# Kod startowy

In [ ]:
######################### NIE ZMIENIAJ TEJ KOMÓRKI ##########################

FINAL_EVALUATION_MODE = False

In [ ]:
######################### NIE ZMIENIAJ TEJ KOMÓRKI ##########################

import os
import torch
import tarfile
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import torchvision.transforms as T

from PIL import Image
from io import BytesIO
from torch.utils.data import Dataset, DataLoader
from torchmetrics.image import PeakSignalNoiseRatio as PSNR
from torchmetrics.image.lpip import LearnedPerceptualImagePatchSimilarity as LPIPS


from stylegan import load_generator
RANDOM_SEED = 1

os.environ["PYTHONHASHSEED"] = str(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

W poniższej komórce znajdują się funkcje oceniające oraz wizualizujące twoje rozwiązania oraz zbiór danych do walidacji.

In [ ]:
######################### NIE ZMIENIAJ TEJ KOMÓRKI ##########################

import math

def round_half_up(number: float) -> int:
    return int(math.floor(number + 0.5))

def plot_batch(batch):
    """ Funkcja wyświetlająca batch zdjęć (wygenerowanych bądź prawdziwych). Maksymalnie pokazuje 8 zdjęć """
    to_show = min(len(batch), 8)

    batch = batch.to('cpu')
    batch = batch * 0.5 + 0.5
    batch = batch.clamp(0, 1)
    batch = batch.permute(0, 2, 3, 1).numpy()

    fig, axes = plt.subplots(1, to_show, figsize=(to_show * 3, 3))

    if to_show == 1:
        axes = [axes]

    for ax, img in zip(axes, batch[:to_show]):
        ax.imshow(img)
        ax.axis('off')
    plt.tight_layout()
    plt.show()


class TarImageDataset(Dataset):
    """ Klasa zbioru danych, używana do walidacji Twojej metody """
    def __init__(self, tar_path):
        self.tar_path = tar_path
        self.tar = tarfile.open(tar_path, 'r')
        self.gt_paths = sorted([m.name for m in self.tar.getmembers() if m.name.startswith('data/GT/') and m.name.endswith('.jpeg')])
        self.gray_paths = [p.replace('GT', 'GRAY') for p in self.gt_paths]

        self.transform = T.Compose([T.ToTensor(), T.Normalize(mean=[0.5], std=[0.5])])

    def __len__(self):
        return len(self.gt_paths)

    def __getitem__(self, idx):
        """ zwraca obrazek monochromatyczny oraz kolorowy """
        gt_member = self.tar.getmember(self.gt_paths[idx])
        gray_member = self.tar.getmember(self.gray_paths[idx])

        gt_image = Image.open(BytesIO(self.tar.extractfile(gt_member).read())).convert('RGB')
        gray_image = Image.open(BytesIO(self.tar.extractfile(gray_member).read())).convert('L')

        return (
            self.transform(gray_image),
            self.transform(gt_image)
        )

    def __del__(self):
        """ Zamyka plik, gdy obiekt tej klasy zostaje usunięty """
        if hasattr(self, 'tar') and self.tar:
            self.tar.close()


def validate_solution(your_model, device, split="val"):
    dataset = TarImageDataset(f"./data/{split}.data")
    dataloader = DataLoader(dataset, batch_size=16, shuffle=False)

    lpips_metric = LPIPS(net_type='alex').to(device)
    psnr_metric = PSNR(data_range=2.0).to(device)

    your_model.eval()

    with torch.no_grad():
        for x_gray, x_gt in dataloader:
            x_gray = x_gray.to(device)
            x_gt = x_gt.to(device)

            x_pred = your_model.predict(x_gray).clamp_(-1, 1)

            psnr_metric.update(x_pred, x_gt)
            lpips_metric.update(x_pred, x_gt)

    avg_psnr = psnr_metric.compute()
    avg_lpips = lpips_metric.compute()

    print("PSNR: ", avg_psnr)
    print("LPIPS:", avg_lpips)

    PSNR_MIN, PSNR_MAX = 22, 26
    LPIPS_MIN, LPIPS_MAX = 0.11, 0.15

    psnr_points = ((torch.clamp(avg_psnr, PSNR_MIN, PSNR_MAX) - PSNR_MIN) / (PSNR_MAX - PSNR_MIN)).item()
    lpips_points = ((LPIPS_MAX - torch.clamp(avg_lpips, LPIPS_MIN, LPIPS_MAX)) / (LPIPS_MAX - LPIPS_MIN)).item()
    total_points = psnr_points * 0.25 + lpips_points * 0.75

    psnr_points = round_half_up(psnr_points * 100)
    lpips_points = round_half_up(lpips_points * 100)
    total_points = round_half_up(total_points * 100)

    return psnr_points, lpips_points, total_points


def show_image_grid(inputs, predictions, targets):
    """
    funkcja wizualizująca wejścia, predykcje oraz obrazy docelowe
    """
    def tensor_to_numpy(img):
        img = (img.clamp(-1, 1) + 1) / 2
        img = img.cpu().numpy()
        if img.shape[0] == 1:
            return img.squeeze(0)
        return np.transpose(img, (1, 2, 0))

    titles = ["Wejścia", "Predykcje modelu", "Obrazy docelowe"]
    images = [inputs, predictions, targets]

    fig, axes = plt.subplots(3, 4, figsize=(16, 10))
    for row in range(3):
        for col in range(4):
            img = tensor_to_numpy(images[row][col])
            cmap = 'gray' if images[row].shape[1] == 1 else None
            axes[row, col].imshow(img, cmap=cmap)
            axes[row, col].axis('off')
        axes[row, 0].text(-0.2, 0.5, titles[row], va='center', ha='right',
                          fontsize=14, transform=axes[row, 0].transAxes)

    plt.suptitle("Przykładowe zdjęcia ze zbioru walidacyjnego")
    plt.tight_layout()
    plt.show()


def show_score_bars(psnr_score, lpips_score, task_score):
    """
    Fukcja wizualizująca uzyskane punkty w formie wykresów słupkowych
    """
    labels = ["punkty za PSNR", "punkty za LPIPS", "punkty za zadanie"][::-1]
    values = [psnr_score, lpips_score, task_score][::-1]

    plt.style.use('ggplot')

    fig, ax = plt.subplots(figsize=(16, 4))
    y = np.arange(len(labels))
    norm = mcolors.Normalize(vmin=0, vmax=100)
    cmap = plt.get_cmap('RdYlGn')
    colors = [cmap(norm(v)) for v in values]

    ax.barh(y, values, color=colors)
    ax.set_xlim(0, 100)
    ax.set_yticks(y)
    ax.set_yticklabels(labels)
    ax.set_xlabel("Punkty")
    ax.set_title("Oceny modelu")
    for i, v in enumerate(values):
        ax.text(v + 1, i, f"{v:.1f}", va='center', fontsize=10)

    plt.tight_layout()
    plt.show()

    # Reset to default style
    plt.style.use('default')


def benchmark_solution(your_model, device, split="val"):
    """
    Funkcja, która waliduje model na danych walidacyjnych i wizualizuje uzyskane rezultaty
    """
    dataset = TarImageDataset(f"./data/{split}.data")
    dataloader = DataLoader(dataset, batch_size=4, shuffle=False)

    x_gray, x_gt = next(iter(dataloader)) 
    x_pred = your_model.predict(x_gray.to(device)).cpu().clamp_(-1, 1)

    psnr_points, lpips_points, total_points = validate_solution(your_model, device, split)
    
    show_image_grid(x_gray, x_pred, x_gt)
    show_score_bars(psnr_points, lpips_points, total_points)



In [ ]:
######################### NIE ZMIENIAJ TEJ KOMÓRKI ##########################

latent_dim = 512   # rozmiar reprezentacji ukrytej modelu StyleGAN
size = 256         # rozdzielczość zdjęć generowanych przez model StyleGAN
device = 'cuda' if torch.cuda.is_available() else 'cpu'

if device == 'cpu':
    print('Uwaga: brak GPU na maszynie!')

# Wczytywanie generatora StyleGAN. 
# Jego implementacja znajduje się w pliku stylegan.py. 
# Dokładna analiza kodu modelu nie jest zabroniona, lecz nie jest sugerowana
generator = load_generator()
generator.to(device)

print('Model został załadowany prawidłowo')

W ponizszej komórce znajduje się kod do generowania twarzy z wykorzystaniem modelu StyleGAN. Generowanie zostało zaprezentowane w dwóch wersjach. Wersja pierwsza jest bardziej szczegółowa i składa się z następujących kroków:

- najpierw generujemy $z$ z rozkładu normalnego
- nastepnie poprzez metodę `get_latent` wykorzystujemy model $f(z)$ do wygenerowania wektorów $w$
- na koniec wywołujemy metodę `style_to_image`, która z wykorzystaniem sieci $g$ generuje zdjęcia twarzy warunkując wektorem $w$.

Alternatywnie możemy wygenerować zdjęcia za pomocą metody `forward`, która łączy ze sobą krok 2 i 3.

In [ ]:
######################### NIE ZMIENIAJ TEJ KOMÓRKI ##########################

if not FINAL_EVALUATION_MODE:
    with torch.no_grad():
        z = torch.randn(8, latent_dim, device=device) 
        w = generator.get_latent(z)
        sample = generator.style_to_image(w)
        plot_batch(sample)

        z = torch.randn(8, latent_dim, device=device)
        sample = generator.forward(z)
        plot_batch(sample)


## Twoje rozwiązanie
W poniższej komórce zaimplementuj swoje rozwiązanie. Upewnij się, że metoda `fit` przygotowuje model koloryzujący zdjęcia, natomaist metoda `predict` wykorzystuje model kolorując dane wejściowe `x_gray`.

In [ ]:
class YourModel(torch.nn.Module):
    def __init__(self, generator):
        super().__init__()
        self.generator = generator

    def fit(self):
        # Tutaj dopasuj metodę koloryzowania zdjęć
        pass

    def predict(self, x_gray):
        # Tutaj wykorzystaj swoją metodę w celu pokolorowania zdjęć x_gray
        return x_gray.repeat(1, 3, 1, 1) # domyślnie zwraca wejściowy batch jako RGB

In [ ]:
######################### NIE ZMIENIAJ TEJ KOMÓRKI ##########################

your_model = YourModel(generator)
your_model.fit()

if not FINAL_EVALUATION_MODE:
    benchmark_solution(your_model, device)